# LIBRARIES

In [3]:
!pip install pandas
!pip install numpy

  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)


In [4]:
import pandas as pd
import numpy as np

# Loading, converting and cleaning of the data

## MARKET TEMPERATURE DATASET

### EDA

In [49]:
df_market_temp = pd.read_csv("dataset/Market_temp_index_uc_month.csv")

In [50]:
df_market_temp

,RegionID,SizeRank,RegionName,RegionType,StateName,2018-01-31,2018-02-28,2018-03-31,2018-04-30,2018-05-31,...,2024-02-29,2024-03-31,2024-04-30,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31,2024-11-30
0,102001,0,United States,country,NaN,50.0,50.0,52.0,54.0,55.0,...,63.0,63.0,62.0,59.0,57.0,55.0,53.0,52.0,51.0,50.0
1,394913,1,"New York, NY",msa,NY,53.0,52.0,55.0,57.0,56.0,...,92.0,90.0,87.0,81.0,78.0,77.0,77.0,74.0,71.0,71.0
2,753899,2,"Los Angeles, CA",msa,CA,69.0,66.0,66.0,67.0,66.0,...,85.0,83.0,80.0,74.0,71.0,67.0,65.0,63.0,62.0,63.0
3,394463,3,"Chicago, IL",msa,IL,48.0,49.0,51.0,52.0,51.0,...,74.0,76.0,75.0,71.0,67.0,64.0,61.0,58.0,56.0,56.0
4,394514,4,"Dallas, TX",msa,TX,55.0,55.0,58.0,60.0,59.0,...,67.0,67.0,64.0,59.0,55.0,52.0,49.0,48.0,49.0,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,753929,935,"Zapata, TX",msa,TX,NaN,NaN,NaN,NaN,NaN,...,39.0,36.0,27.0,40.0,53.0,47.0,31.0,34.0,42.0,41.0
924,394743,936,"Ketchikan, AK",msa,AK,37.0,35.0,34.0,37.0,49.0,...,74.0,69.0,49.0,53.0,61.0,66.0,57.0,53.0,44.0,50.0
925,753874,937,"Craig, CO",msa,CO,3.0,23.0,52.0,67.0,56.0,...,61.0,55.0,59.0,57.0,48.0,33.0,21.0,28.0,41.0,37.0
926,395188,938,"Vernon, TX",msa,TX,51.0,45.0,NaN,48.0,65.0,...,83.0,78.0,64.0,65.0,61.0,52.0,57.0,56.0,51.0,43.0


We can see that market temperature dataset has 928 rows and 88 columns. First 5 columns are:

- ***RegionID:*** Unique identifier. Each US region has an id. 102001 is United States id, not a specific country. There 928 different IDs, except 102001, all other corrispond to a specific region. 
- ***SizeRank:*** Ranking of the region from most populated to the less populated. SizeRank has 925 different values. It means that some of countries have the same rank. They are not NaN values (except United States obviously), I won't "fix" it since it is not a specific region.
- ***RegionName:*** Associates a name to id number. 928 different Region names obviously. 
- ***RegionType:*** 2 different types of regions:
    - **country:** Since we have a row about the United States, here the country is only for US. 
    - **msa:** Metropolitan Statistical Area (all 927 remaining rows).
- ***StateName:*** The name of each state (US of course is NaN)


**Date columns:**<br>
As we can see the other 83 remaining columnns are dates. Pandas sees those dates as a column name (string) not a date sequence. So, I need to transform them to do time series analysis. 

In [51]:
df_market_temp['RegionID'].count()


np.int64(928)

In [52]:
len(df_market_temp['SizeRank'].unique())

925

In [53]:
df_market_temp.isnull().sum()

RegionID       0
SizeRank       0
RegionName     0
RegionType     0
StateName      1
              ..
2024-07-31     1
2024-08-31     5
2024-09-30     7
2024-10-31     8
2024-11-30    13
Length: 88, dtype: int64

In [54]:
len(df_market_temp['RegionName'].unique())

928

In [55]:
df_market_temp['RegionType'].unique()

array(['country', 'msa'], dtype=object)

In [56]:
df_market_temp[df_market_temp['RegionType']=='country']

,RegionID,SizeRank,RegionName,RegionType,StateName,2018-01-31,2018-02-28,2018-03-31,2018-04-30,2018-05-31,...,2024-02-29,2024-03-31,2024-04-30,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31,2024-11-30
0,102001,0,United States,country,NaN,50.0,50.0,52.0,54.0,55.0,...,63.0,63.0,62.0,59.0,57.0,55.0,53.0,52.0,51.0,50.0


In [57]:
len(df_market_temp[df_market_temp['RegionType']=='msa'])

927

In [58]:
df_market_temp['StateName'].isnull().sum()

np.int64(1)

### Reshaping the dataset

As we can see the other 83 remaining columnns are dates. Pandas sees those dates as a column name (string) not a date sequence. So, I need to transform them to do time series analysis. 

I use pandas melt function for this operation

In [62]:
df_market_temp

,RegionID,SizeRank,RegionName,RegionType,StateName,2018-01-31,2018-02-28,2018-03-31,2018-04-30,2018-05-31,...,2024-02-29,2024-03-31,2024-04-30,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31,2024-11-30
0,102001,0,United States,country,NaN,50.0,50.0,52.0,54.0,55.0,...,63.0,63.0,62.0,59.0,57.0,55.0,53.0,52.0,51.0,50.0
1,394913,1,"New York, NY",msa,NY,53.0,52.0,55.0,57.0,56.0,...,92.0,90.0,87.0,81.0,78.0,77.0,77.0,74.0,71.0,71.0
2,753899,2,"Los Angeles, CA",msa,CA,69.0,66.0,66.0,67.0,66.0,...,85.0,83.0,80.0,74.0,71.0,67.0,65.0,63.0,62.0,63.0
3,394463,3,"Chicago, IL",msa,IL,48.0,49.0,51.0,52.0,51.0,...,74.0,76.0,75.0,71.0,67.0,64.0,61.0,58.0,56.0,56.0
4,394514,4,"Dallas, TX",msa,TX,55.0,55.0,58.0,60.0,59.0,...,67.0,67.0,64.0,59.0,55.0,52.0,49.0,48.0,49.0,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
923,753929,935,"Zapata, TX",msa,TX,NaN,NaN,NaN,NaN,NaN,...,39.0,36.0,27.0,40.0,53.0,47.0,31.0,34.0,42.0,41.0
924,394743,936,"Ketchikan, AK",msa,AK,37.0,35.0,34.0,37.0,49.0,...,74.0,69.0,49.0,53.0,61.0,66.0,57.0,53.0,44.0,50.0
925,753874,937,"Craig, CO",msa,CO,3.0,23.0,52.0,67.0,56.0,...,61.0,55.0,59.0,57.0,48.0,33.0,21.0,28.0,41.0,37.0
926,395188,938,"Vernon, TX",msa,TX,51.0,45.0,NaN,48.0,65.0,...,83.0,78.0,64.0,65.0,61.0,52.0,57.0,56.0,51.0,43.0


In [63]:
df_market_temp_copy = df_market_temp.copy()

# pivoting all date columns into rows
df_market_temp_copy = pd.melt(
    df_market_temp,
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], # keep the columns as they are
    var_name='Date',           # the new column that holds dates that were columns
    value_name='MarketTemperature'  # the new column that holds the temperature values, now we have 88 rows for each date and we can see the temperature 
                                    # values here  not like first version that for each region there were 88 columns that holds these temperatures. 
)

# convert columns into pandas timestamp 
df_market_temp_copy['Date'] = pd.to_datetime(df_market_temp_copy['Date'])

# make Date the row index, not just a regular column
df_market_temp_copy.set_index('Date', inplace=True)

In [64]:
df_market_temp_copy

,RegionID,SizeRank,RegionName,RegionType,StateName,MarketTemperature
Date,,,,,,
2018-01-31,102001,0,United States,country,NaN,50.0
2018-01-31,394913,1,"New York, NY",msa,NY,53.0
2018-01-31,753899,2,"Los Angeles, CA",msa,CA,69.0
2018-01-31,394463,3,"Chicago, IL",msa,IL,48.0
2018-01-31,394514,4,"Dallas, TX",msa,TX,55.0
...,...,...,...,...,...,...
2024-11-30,753929,935,"Zapata, TX",msa,TX,41.0
2024-11-30,394743,936,"Ketchikan, AK",msa,AK,50.0
2024-11-30,753874,937,"Craig, CO",msa,CO,37.0


In [69]:
df_market_temp_copy.columns

Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       'MarketTemperature'],
      dtype='object')

In [70]:
df_market_temp_copy.index

DatetimeIndex(['2018-01-31', '2018-01-31', '2018-01-31', '2018-01-31',
               '2018-01-31', '2018-01-31', '2018-01-31', '2018-01-31',
               '2018-01-31', '2018-01-31',
               ...
               '2024-11-30', '2024-11-30', '2024-11-30', '2024-11-30',
               '2024-11-30', '2024-11-30', '2024-11-30', '2024-11-30',
               '2024-11-30', '2024-11-30'],
              dtype='datetime64[ns]', name='Date', length=77024, freq=None)

In [68]:
len(df_market_temp_copy.index.unique())

83

### Market temprature index data cleaning:

- missing values: 
    - in *StateName* there are 83 missing values, but we already know that the United State has a row by its own but don't have anything in StateName, after checking I saw that there is no any region with state name "US" or "us" so instead of deleting United States from the table I will just add "US" as its StateName. 
    - There are other missing values in the *MarketTemperature* column. But, here the problem is not easy as in the StateName. So, I will use ***forward fill - ffill*** method that we saw during the lecture. Since pandas scan each column from top to bottom (earliest date to latest date) the index moves in chronological order and when it hits a NaN, ffill method looks backwards and copies the last known real value. The problem with this method can be the very first value is NaN, in that case I can use ***bfill*** method, which propagates the next known valid value backward to replace missing values.
- outliers: if there are any

In [87]:
df_market_temp_copy.isnull().sum()

RegionID                0
SizeRank                0
RegionName              0
RegionType              0
StateName              83
MarketTemperature    2220
dtype: int64

In [ ]:
df_market_temp_copy['StateName'].isnull()

Date
2018-01-31     True
2018-01-31    False
2018-01-31    False
2018-01-31    False
2018-01-31    False
              ...  
2024-11-30    False
2024-11-30    False
2024-11-30    False
2024-11-30    False
2024-11-30    False
Name: StateName, Length: 77024, dtype: bool

In [97]:
df_market_temp_copy[
    df_market_temp_copy['StateName'].isna() |
    (df_market_temp_copy['StateName'] == 'NaN')
]

,RegionID,SizeRank,RegionName,RegionType,StateName,MarketTemperature
Date,,,,,,
2018-01-31,102001,0,United States,country,NaN,50.0
2018-02-28,102001,0,United States,country,NaN,50.0
2018-03-31,102001,0,United States,country,NaN,52.0
2018-04-30,102001,0,United States,country,NaN,54.0
2018-05-31,102001,0,United States,country,NaN,55.0
...,...,...,...,...,...,...
2024-07-31,102001,0,United States,country,NaN,55.0
2024-08-31,102001,0,United States,country,NaN,53.0
2024-09-30,102001,0,United States,country,NaN,52.0


In [92]:
df_market_temp_copy[df_market_temp_copy['StateName']=="us"]

,RegionID,SizeRank,RegionName,RegionType,StateName,MarketTemperature
Date,,,,,,


In [98]:
df_market_temp_copy['StateName'] = df_market_temp_copy['StateName'].fillna('US')

In [104]:
df_market_temp_copy.isnull().sum()

RegionID                0
SizeRank                0
RegionName              0
RegionType              0
StateName               0
MarketTemperature    2220
dtype: int64

In [103]:
df_market_temp_copy[
    df_market_temp_copy['MarketTemperature'].isna() |
    (df_market_temp_copy['MarketTemperature'] == 'NaN')
]


,RegionID,SizeRank,RegionName,RegionType,StateName,MarketTemperature
Date,,,,,,
2018-01-31,786258,483,"Fort Payne, AL",msa,AL,NaN
2018-01-31,845162,535,"Granbury, TX",msa,TX,NaN
2018-01-31,786250,595,"Alexander City, AL",msa,AL,NaN
2018-01-31,394538,596,"Douglas, GA",msa,GA,NaN
2018-01-31,394987,635,"Plymouth, IN",msa,IN,NaN
...,...,...,...,...,...,...
2024-11-30,394947,762,"Ottumwa, IA",msa,IA,NaN
2024-11-30,394331,774,"Angola, IN",msa,IN,NaN
2024-11-30,394777,776,"Laurinburg, NC",msa,NC,NaN


Market temperature missing values fix

In [ ]:
# sort first so ffill travels in chronological order
df_market_temp_copy = df_market_temp_copy.sort_index()

df_market_temp_copy['MarketTemperature'] = (
    df_market_temp_copy.groupby('StateName')['MarketTemperature'] # Grouping is important to avoid that first NaN value of a state uses the last value of previous state.
    .ffill()   # fill gaps using last known value within same state
    .bfill()   # catch any NaNs at the very start of a state's history
)

print(df_market_temp_copy['MarketTemperature'].isna().sum())  # should print 0

0


In [107]:
df_market_temp_copy.isnull().sum()

RegionID             0
SizeRank             0
RegionName           0
RegionType           0
StateName            0
MarketTemperature    0
dtype: int64

In [110]:
df_market_temp_copy

,RegionID,SizeRank,RegionName,RegionType,StateName,MarketTemperature
Date,,,,,,
2018-01-31,102001,0,United States,country,US,50.0
2018-01-31,394913,1,"New York, NY",msa,NY,53.0
2018-01-31,753899,2,"Los Angeles, CA",msa,CA,69.0
2018-01-31,394463,3,"Chicago, IL",msa,IL,48.0
2018-01-31,394514,4,"Dallas, TX",msa,TX,55.0
...,...,...,...,...,...,...
2024-11-30,753929,935,"Zapata, TX",msa,TX,41.0
2024-11-30,394743,936,"Ketchikan, AK",msa,AK,50.0
2024-11-30,753874,937,"Craig, CO",msa,CO,37.0


## MEDIAN SALE PRICE

In [77]:
df_sale_price = pd.read_csv("dataset/Median_sale_price_uc_month.csv")

In [78]:
df_sale_price

,RegionID,SizeRank,RegionName,RegionType,StateName,2008-02-29,2008-03-31,2008-04-30,2008-05-31,2008-06-30,...,2024-01-31,2024-02-29,2024-03-31,2024-04-30,2024-05-31,2024-06-30,2024-07-31,2024-08-31,2024-09-30,2024-10-31
0,102001,0,United States,country,NaN,170922.0,175612.0,177500.0,180000.0,185000.0,...,324000.0,335000.0,345000.0,350000.0,360000.0,369000.0,361000.0,360000.0,350500.0,355000.0
1,394913,1,"New York, NY",msa,NY,397000.0,390000.0,390000.0,390000.0,399900.0,...,570000.0,565000.0,575000.0,585000.0,615000.0,640000.0,649450.0,640000.0,625000.0,624451.0
2,753899,2,"Los Angeles, CA",msa,CA,470000.0,455000.0,457000.0,440000.0,435000.0,...,865000.0,915000.0,916250.0,950000.0,970000.0,960000.0,950500.0,932000.0,915000.0,941000.0
3,394463,3,"Chicago, IL",msa,IL,216750.0,220000.0,221000.0,227000.0,235000.0,...,279000.0,285000.0,300000.0,315000.0,325000.0,337275.0,330000.0,320000.0,317881.0,315000.0
4,394514,4,"Dallas, TX",msa,TX,138000.0,146000.0,144950.0,150000.0,156000.0,...,368000.0,380000.0,390000.0,400000.0,405000.0,400000.0,395000.0,390000.0,380000.0,384000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
708,394869,869,"Moberly, MO",msa,MO,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,139000.0,152000.0,163250.0,180500.0,186000.0,46338.0,185900.0
709,394371,891,"Beatrice, NE",msa,NE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,200000.0
710,753914,908,"Port Lavaca, TX",msa,TX,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,177884.0,209255.0,103075.0,140948.0
711,395003,912,"Price, UT",msa,UT,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,228500.0,305000.0


In [79]:
df_sale_price.columns

Index(['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName',
       '2008-02-29', '2008-03-31', '2008-04-30', '2008-05-31', '2008-06-30',
       ...
       '2024-01-31', '2024-02-29', '2024-03-31', '2024-04-30', '2024-05-31',
       '2024-06-30', '2024-07-31', '2024-08-31', '2024-09-30', '2024-10-31'],
      dtype='object', length=206)

Same problem as previous, dates are columns also here. So I will apply the same method to make those date indexes. 

In [80]:
df_sale_price_copy = df_sale_price.copy()

df_sale_price_copy = pd.melt(
    df_sale_price,
    id_vars=['RegionID', 'SizeRank', 'RegionName', 'RegionType', 'StateName'], 
    var_name='Date',          
    value_name='MedianSalePrice'  
)

df_sale_price_copy['Date'] = pd.to_datetime(df_sale_price_copy['Date'])

df_sale_price_copy.set_index('Date', inplace=True)

In [81]:
df_sale_price_copy

,RegionID,SizeRank,RegionName,RegionType,StateName,MedianSalePrice
Date,,,,,,
2008-02-29,102001,0,United States,country,NaN,170922.0
2008-02-29,394913,1,"New York, NY",msa,NY,397000.0
2008-02-29,753899,2,"Los Angeles, CA",msa,CA,470000.0
2008-02-29,394463,3,"Chicago, IL",msa,IL,216750.0
2008-02-29,394514,4,"Dallas, TX",msa,TX,138000.0
...,...,...,...,...,...,...
2024-10-31,394869,869,"Moberly, MO",msa,MO,185900.0
2024-10-31,394371,891,"Beatrice, NE",msa,NE,200000.0
2024-10-31,753914,908,"Port Lavaca, TX",msa,TX,140948.0


### Median sale price Datasets cleaning: 

- missing values: 
    - StateName: same as market temperature dataset. I am going to fill missing StateName for United States with "US"
    - MedianSalePrice: by calculating 27447 missing values over total 143313, we can see that almost 20% of total data are missing, this gap is huge. To handle this missing values I am going to use the same methods. ***ffill*** and ***bfill*** as before. 
- outliers

In [108]:
df_sale_price_copy.isnull().sum()

RegionID               0
SizeRank               0
RegionName             0
RegionType             0
StateName            201
MedianSalePrice    27447
dtype: int64

In [109]:
df_sale_price_copy[
    df_sale_price_copy['StateName'].isna() |
    (df_sale_price_copy['StateName'] == 'NaN')
]

,RegionID,SizeRank,RegionName,RegionType,StateName,MedianSalePrice
Date,,,,,,
2008-02-29,102001,0,United States,country,NaN,170922.0
2008-03-31,102001,0,United States,country,NaN,175612.0
2008-04-30,102001,0,United States,country,NaN,177500.0
2008-05-31,102001,0,United States,country,NaN,180000.0
2008-06-30,102001,0,United States,country,NaN,185000.0
...,...,...,...,...,...,...
2024-06-30,102001,0,United States,country,NaN,369000.0
2024-07-31,102001,0,United States,country,NaN,361000.0
2024-08-31,102001,0,United States,country,NaN,360000.0


In [111]:
df_sale_price_copy['StateName'] = df_sale_price_copy['StateName'].fillna('US')

In [112]:
df_sale_price_copy

,RegionID,SizeRank,RegionName,RegionType,StateName,MedianSalePrice
Date,,,,,,
2008-02-29,102001,0,United States,country,US,170922.0
2008-02-29,394913,1,"New York, NY",msa,NY,397000.0
2008-02-29,753899,2,"Los Angeles, CA",msa,CA,470000.0
2008-02-29,394463,3,"Chicago, IL",msa,IL,216750.0
2008-02-29,394514,4,"Dallas, TX",msa,TX,138000.0
...,...,...,...,...,...,...
2024-10-31,394869,869,"Moberly, MO",msa,MO,185900.0
2024-10-31,394371,891,"Beatrice, NE",msa,NE,200000.0
2024-10-31,753914,908,"Port Lavaca, TX",msa,TX,140948.0


In [113]:
df_sale_price_copy[
    df_sale_price_copy['MedianSalePrice'].isna() |
    (df_sale_price_copy['MedianSalePrice'] == 'NaN')
]

,RegionID,SizeRank,RegionName,RegionType,StateName,MedianSalePrice
Date,,,,,,
2008-02-29,845159,86,"Poughkeepsie, NY",msa,NY,NaN
2008-02-29,394421,130,"Brownsville, TX",msa,TX,NaN
2008-02-29,395197,178,"Waco, TX",msa,TX,NaN
2008-02-29,395103,182,"Sioux Falls, SD",msa,SD,NaN
2008-02-29,394772,187,"Laredo, TX",msa,TX,NaN
...,...,...,...,...,...,...
2024-08-31,394371,891,"Beatrice, NE",msa,NE,NaN
2024-08-31,395003,912,"Price, UT",msa,UT,NaN
2024-09-30,394343,521,"Athens, OH",msa,OH,NaN


In [122]:
pricdf_sale_price_copye_df = df_sale_price_copy.sort_index()

df_sale_price_copy['MedianSalePrice'] = (
    df_sale_price_copy.groupby('StateName')['MedianSalePrice']
    .ffill()
    .bfill()
)

print(df_sale_price_copy['MedianSalePrice'].isna().sum())  # should print 0


0


In [124]:
df_sale_price_copy.isnull().sum()

RegionID           0
SizeRank           0
RegionName         0
RegionType         0
StateName          0
MedianSalePrice    0
dtype: int64